## Instalando librerías

In [0]:
from pyspark.sql.functions import *
from delta.tables import DeltaTable

## Ingestando en Bronze Data

In [0]:
dbutils.widgets.removeAll()

### Definiendo y obteniendo los widgets

In [0]:
%sql
CREATE WIDGET TEXT catalog_name  DEFAULT 'smartdata_project';
CREATE WIDGET TEXT schema_sink DEFAULT 'bronze';
CREATE WIDGET TEXT schema_source DEFAULT 'raw';
CREATE WIDGET TEXT source_product DEFAULT 'federated_sql_catalog.source.dim_products';
CREATE WIDGET TEXT sink_product DEFAULT 'bronze_products';
CREATE WIDGET TEXT source_person DEFAULT 'federated_sql_catalog.source.dim_persons';
CREATE WIDGET TEXT sink_person DEFAULT 'bronze_persons';
CREATE WIDGET TEXT source_sales DEFAULT 'abfss://raw@adlssmartdataeastus2001.dfs.core.windows.net/sales/';
CREATE WIDGET TEXT sink_sales DEFAULT 'bronze_sales';

In [0]:
catalog_name = dbutils.widgets.get('catalog_name')
schema_sink = dbutils.widgets.get('schema_sink')
schema_source = dbutils.widgets.get('schema_source')
source_product = dbutils.widgets.get('source_product')
sink_product = dbutils.widgets.get('sink_product')
source_person = dbutils.widgets.get('source_person')
sink_person = dbutils.widgets.get('sink_person')
source_sales = dbutils.widgets.get('source_sales')
sink_sales = dbutils.widgets.get('sink_sales')

### Ingestando Bronze Sales

In [0]:
location = f"/Volumes/{catalog_name}/{schema_source}/autoloader/checkpoint/"

df = spark.readStream.format("cloudFiles") \
    .option("cloudFiles.format", "csv") \
    .option('CloudFiles.schemaEvolutionMode', 'rescue') \
    .option("cloudFiles.schemaLocation", location) \
    .load(source_sales) \
    .withColumn("_source_file", col('_metadata.file_name'))\
    .withColumn('updated_at', to_timestamp(col('end_at'), 'dd/MM/yyyy H:mm'))\
    .withColumn("_ingestion_timestamp", current_timestamp())
    
df.writeStream \
    .option("checkpointLocation", location) \
    .outputMode('append') \
    .trigger(availableNow=True) \
    .option('mergeSchema', True) \
    .toTable(f"{catalog_name}.{schema_sink}.{sink_sales}")

### Obteniendo Watermarks

In [0]:
watermarks = spark.read.format('json').option('multiline', True).load(f'/Volumes/{catalog_name}/{schema_source}/autoloader/watermarks.json')

In [0]:
w_products = watermarks.filter(col('table') == 'products').select('last_updated').collect()[0]['last_updated']
w_persons = watermarks.filter(col('table') == 'persons').select('last_updated').collect()[0]['last_updated']
w_sales = watermarks.filter(col('table') == 'sales').select('last_updated').collect()[0]['last_updated']

### Ingestando Bronze Products

In [0]:
target_product_table = f'{catalog_name}.{schema_sink}.{sink_product}'
if spark.catalog.tableExists(target_product_table):

    source_table = DeltaTable.forName(spark, target_product_table)
    df_product = spark.sql(f"SELECT * FROM {source_product} WHERE updated_at > '{w_products}'")
    df_product = df_product.withColumn('_ingestion_timestamp', current_timestamp())

    (
        source_table.alias("target")
        .merge(
            df_product.alias("source"),
            f"target.product_id = source.product_id"
        )
        # Registro existe y cambió algo → actualizar
        .whenMatchedUpdateAll()
        # Registro nuevo → insertar
        .whenNotMatchedInsertAll()
        .execute()
    )

else:
    df_product = spark.table(source_product)
    (
        df_product.write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .clusterBy('updated_at', "age")  # Liquid Clustering
        .saveAsTable(target_product_table)
    )

### Ingestando Bronze Persons

In [0]:
target_person_table = f'{catalog_name}.{schema_sink}.{sink_person}'
if spark.catalog.tableExists(target_person_table):

    source_table = DeltaTable.forName(spark, target_person_table)
    df_person = spark.sql(f"SELECT * FROM {source_person} WHERE updated_at > '{w_persons}'")
    df_person = df_person.withColumn('_ingestion_timestamp', current_timestamp())

    (
        source_table.alias("target")
        .merge(
            df_person.alias("source"),
            f"target.customer_id_original = source.customer_id_original"
        )
        # Registro existe y cambió algo → actualizar
        .whenMatchedUpdateAll()
        # Registro nuevo → insertar
        .whenNotMatchedInsertAll()
        .execute()
    )

else:
    df_person = spark.table(source_person)
    (
        df_person.write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .clusterBy('updated_at', "age")  # Liquid Clustering
        .saveAsTable(target_person_table)
    )

In [0]:
dbutils.jobs.taskValues.set('w_products',w_products)
dbutils.jobs.taskValues.set('w_persons',w_persons)
dbutils.jobs.taskValues.set('w_sales',w_sales)